# L4: Attention Mechanisms - The Heart of Transformers

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 4 of 15

---

## 📚 Learning Objectives

By the end of this lesson, you will:
- Understand why attention is crucial for transformers
- Learn the Query-Key-Value (QKV) mechanism
- Implement self-attention from scratch
- Explore multi-head attention
- Visualize attention patterns

---

## 🧠 Concept: What is Attention?

**Attention** allows models to focus on relevant parts of the input when processing each element.

### Analogy: Reading Comprehension
When answering "What did the king do?", you:
1. Focus on words related to "king"
2. Pay attention to verbs near "king"
3. Ignore irrelevant parts

This is exactly what attention does!

### Why Attention?
**Before Attention (RNNs):**
- Process sequentially: word1 → word2 → word3
- Information bottleneck
- Long-range dependencies are hard

**With Attention (Transformers):**
- Process all words simultaneously
- Each word can attend to any other word
- Captures long-range dependencies easily

---

In [ ]:
# Step 1: Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

## 📖 Part 1: Understanding Query, Key, Value (QKV)

Attention uses three components:

1. **Query (Q):** "What am I looking for?"
2. **Key (K):** "What do I contain?"
3. **Value (V):** "What information do I have?"

### Analogy: Library Search
- **Query:** Your search term ("machine learning books")
- **Key:** Book titles/keywords
- **Value:** Actual book content

Process:
1. Compare Query with all Keys (find relevant books)
2. Get attention scores (relevance scores)
3. Retrieve Values weighted by scores (read relevant books)

### Mathematical Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $QK^T$: Compute similarity between queries and keys
- $\sqrt{d_k}$: Scaling factor (prevents large values)
- $\text{softmax}$: Convert to probabilities
- Multiply by $V$: Get weighted values

---

In [ ]:
# Step 2: Implement Scaled Dot-Product Attention

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Query matrix (batch_size, seq_len, d_k)
        K: Key matrix (batch_size, seq_len, d_k)
        V: Value matrix (batch_size, seq_len, d_v)
        mask: Optional mask (batch_size, seq_len, seq_len)
    
    Returns:
        output: Attention output (batch_size, seq_len, d_v)
        attention_weights: Attention scores (batch_size, seq_len, seq_len)
    """
    # Get dimension of keys
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores (QK^T)
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq_len, seq_len)
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    # Step 3: Apply mask (if provided)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Step 4: Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Multiply by values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

print("✅ Scaled dot-product attention implemented!")

In [ ]:
# Step 3: Test with Simple Example

# Create simple input
# Sentence: "The cat sat"
seq_len = 3
d_model = 4  # Embedding dimension

# Random embeddings for demonstration
x = torch.randn(1, seq_len, d_model)  # (batch=1, seq_len=3, d_model=4)

# For self-attention, Q, K, V come from the same input
Q = K = V = x

# Compute attention
output, attention_weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nAttention weights:\n{attention_weights[0]}")
print(f"\nSum of attention weights (should be 1.0): {attention_weights[0].sum(dim=-1)}")

## 🎨 Part 2: Visualizing Attention

Let's visualize how attention works with a real sentence.

---

In [ ]:
# Step 4: Visualize Attention Patterns

def visualize_attention(attention_weights, tokens, title="Attention Weights"):
    """
    Visualize attention weights as a heatmap.
    """
    plt.figure(figsize=(10, 8))
    
    # Convert to numpy
    attn = attention_weights.detach().numpy()
    
    # Create heatmap
    sns.heatmap(
        attn,
        xticklabels=tokens,
        yticklabels=tokens,
        annot=True,
        fmt='.3f',
        cmap='YlOrRd',
        cbar_kws={'label': 'Attention Weight'}
    )
    
    plt.xlabel('Key (attending to)')
    plt.ylabel('Query (attending from)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Example sentence
tokens = ["The", "cat", "sat"]
visualize_attention(attention_weights[0], tokens)

## 🔢 Part 3: Self-Attention Layer

In practice, we use learnable weight matrices to project inputs into Q, K, V:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

---

In [ ]:
# Step 5: Implement Self-Attention Layer

class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        """
        Self-Attention layer.
        
        Args:
            d_model: Input/output dimension
            d_k: Dimension of queries and keys
        """
        super(SelfAttention, self).__init__()
        
        self.d_k = d_k
        
        # Learnable weight matrices
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)
        
        # Output projection
        self.W_o = nn.Linear(d_k, d_model, bias=False)
    
    def forward(self, x, mask=None):
        """
        Forward pass.
        
        Args:
            x: Input tensor (batch_size, seq_len, d_model)
            mask: Optional mask
        
        Returns:
            output: Attention output
            attention_weights: Attention scores
        """
        # Project to Q, K, V
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Apply attention
        attn_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Project back to d_model
        output = self.W_o(attn_output)
        
        return output, attention_weights

# Test the layer
d_model = 512
d_k = 64
batch_size = 2
seq_len = 10

self_attn = SelfAttention(d_model, d_k)
x = torch.randn(batch_size, seq_len, d_model)
output, attn_weights = self_attn(x)

print(f"✅ Self-Attention layer created!")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

## 🎯 Part 4: Multi-Head Attention

**Multi-head attention** runs multiple attention mechanisms in parallel:

### Why Multiple Heads?
- Different heads can focus on different aspects
- Head 1: Syntactic relationships
- Head 2: Semantic relationships
- Head 3: Long-range dependencies

### Process
1. Split into multiple heads
2. Apply attention to each head
3. Concatenate results
4. Project back to original dimension

---

In [ ]:
# Step 6: Implement Multi-Head Attention

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        Multi-Head Attention.
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
        """
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Weight matrices for all heads (combined)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        """Split into multiple heads."""
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    
    def combine_heads(self, x):
        """Combine multiple heads."""
        batch_size, num_heads, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, x, mask=None):
        batch_size = x.size(0)
        
        # Project and split into heads
        Q = self.split_heads(self.W_q(x))  # (batch, num_heads, seq_len, d_k)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))
        
        # Apply attention for each head
        attn_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads
        attn_output = self.combine_heads(attn_output)
        
        # Final projection
        output = self.W_o(attn_output)
        
        return output, attention_weights

# Test Multi-Head Attention
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
output, attn_weights = mha(x)

print(f"✅ Multi-Head Attention created!")
print(f"Number of heads: {num_heads}")
print(f"Dimension per head: {d_model // num_heads}")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

## 🎓 Key Takeaways

1. **Attention** allows models to focus on relevant parts of input
2. **QKV mechanism:** Query what you need, match with Keys, retrieve Values
3. **Scaled dot-product:** Prevents large values that hurt training
4. **Self-attention:** Each position attends to all positions
5. **Multi-head attention:** Multiple attention patterns in parallel
6. **Transformers:** Built entirely on attention (no recurrence!)

---

## 🔬 Exercises

1. Implement masked attention (for causal language modeling)
2. Visualize attention patterns for different sentences
3. Compare single-head vs multi-head attention
4. Experiment with different numbers of heads
5. Implement cross-attention (Q from one source, K,V from another)

---

## 📚 Additional Resources

- **Paper:** "Attention Is All You Need" (Vaswani et al., 2017)
- **Blog:** [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- **Video:** [Attention Mechanism Explained](https://www.youtube.com/watch?v=XowwKOAWYoQ)
- **Interactive:** [Attention Visualization Tool](https://github.com/jessevig/bertviz)

---

## ➡️ Next Lesson

**L5: Positional Encoding** - Learn how transformers understand word order

---

**Happy Learning! 🚀**